# IndicOOC — MVP Setup Notebook (Step 1 of the build)

This is the first notebook in the IndicOOC build. Its only job is to get your environment ready and *prove* it works — no project data yet, no baseline logic yet. That comes next.

**What this notebook does, in order:**
1. Confirm you have a free GPU attached
2. Mount Google Drive so nothing you build disappears when Colab disconnects
3. Create a project folder structure in Drive
4. Install the Python packages the project needs
5. Load a small vision-language model (SigLIP) as a sanity check
6. Run one toy image + two captions through it, to see the exact signal your OOC detector will eventually threshold

**Why start here instead of jumping to real data:** Objective O2 in your synopsis calls for a CLIP/SigLIP zero-shot baseline. Before collecting a single annotated example (O1), it's worth confirming that baseline actually runs in a free Colab session — if it doesn't fit or is too slow, you want to know in the first ten minutes, not in week 6.

Run the cells top to bottom. Markdown cells explain the *why*; code cells do the *what*.

## Step 1 — Turn on a GPU runtime

Colab notebooks default to CPU-only, which makes even small model loads painfully slow.

Go to: **Runtime → Change runtime type → Hardware accelerator → T4 GPU → Save**

The free tier typically hands you a T4 GPU (16 GB VRAM), but Google doesn't guarantee a specific GPU or session length — quotas are dynamic, sessions cap out around 12 hours, and weekly GPU hours are limited. That's still plenty for SigLIP/CLIP-sized models (a few hundred million parameters). It is **not** enough to comfortably run the 7B-parameter Indic VLMs (Chitrarth-1, Maya) from Objective O2 baseline (iv) — those need 4-bit quantization and, per your own feasibility table, the institute GPU lab or Colab Pro.

Run the cell below to confirm what actually got attached.

In [ ]:
import torch

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    total_vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"VRAM: {total_vram_gb:.1f} GB")
else:
    print("No GPU found. Go to Runtime > Change runtime type > T4 GPU, "
          "then Runtime > Restart session, then re-run this cell.")

## Step 2 — Mount Google Drive

Colab's local disk (`/content/`) is wiped every time the runtime disconnects. Over the coming weeks you'll accumulate a dataset, annotation files and model checkpoints — none of that can live only in `/content/`.

Mounting Drive gives you a persistent folder at `/content/drive/MyDrive/` that survives across sessions. Everything the project creates from here on gets saved there.

Running the next cell pops up a Google auth flow — sign in with the account whose Drive you want to use, and allow access.

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

## Step 3 — Create the project folder structure

One folder per concern, matching how the objectives in your synopsis are organized:

| Folder | Will hold |
|---|---|
| `data/images` | Downloaded / sample images |
| `data/annotations` | Your (image, claim, label) triples — csv or jsonl |
| `models` | Local checkpoints or LoRA adapters (for O3, later) |
| `notebooks` | Copies of your Colab notebooks, so nothing is orphaned in "Untitled0.ipynb" |
| `outputs` | Metrics, confusion matrices, logs, figures |

Keeping this consistent now saves you from a messy `MyDrive/` root by Review 1.

In [ ]:
import os

PROJECT_ROOT = '/content/drive/MyDrive/IndicOOC'

folders = [
    'data/images',
    'data/annotations',
    'models',
    'notebooks',
    'outputs',
]

for folder in folders:
    os.makedirs(os.path.join(PROJECT_ROOT, folder), exist_ok=True)

print("Project root:", PROJECT_ROOT)
for folder in folders:
    print(" -", folder)

## Step 4 — Install project dependencies

Colab ships with `torch` pre-installed, but a few packages this project needs aren't there by default:

- `transformers` — loads CLIP, SigLIP, MuRIL, IndicBERT and (later) the VLMs, all through one API
- `accelerate` — lets `transformers` place models on GPU efficiently, and is a prerequisite for 4-bit loading later
- `pillow` — image loading/handling
- `pandas` — for the (image, claim, label) tables you'll build in O1
- `scikit-learn` — macro-F1, per-class F1, confusion matrices: exactly the metrics your objectives call for
- `sentencepiece` — tokenizer backend several multilingual models need

`-q` just keeps the install output quiet.

In [ ]:
!pip install -q transformers accelerate pillow pandas scikit-learn sentencepiece
print("Done.")

## Step 5 — Load a real model: SigLIP

Time to prove the environment can actually run one of your O2 baseline families.

We'll use `google/siglip-base-patch16-224` — the SigLIP family is the newer, generally stronger sibling of CLIP for exactly this kind of image–text similarity scoring, and the base size (~200M params) downloads in under a minute and fits on a T4 with room to spare.

This is the same *mechanism* behind Objective O2 baseline (i): "CLIP/SigLIP similarity thresholding." Later we'll wrap this in a proper threshold + evaluation loop; right now we just want to see it work.

In [ ]:
from transformers import AutoProcessor, AutoModel
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
model_name = "google/siglip-base-patch16-224"

processor = AutoProcessor.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name).to(device)
model.eval()

print(f"Loaded {model_name} on {device}")

## Step 6 — Run one toy example end to end

This uses a public demo image (the standard COCO image used in HuggingFace's own CLIP/SigLIP examples) and two throwaway captions — **not** real project data. The point is only to confirm the full path works: image in, text in, a score out.

One caption matches the image, one doesn't. If the environment is working, the model should score the matching caption much higher. (If this particular URL ever breaks, swap in any public image URL — the mechanics are what matter, not this specific picture.)

In [ ]:
import requests
from PIL import Image

# Public demo image (two cats on a couch) — swap for a real IndicOOC example once you start O1
image_url = "http://images.cocodataset.org/val2017/000000039769.jpg"
image = Image.open(requests.get(image_url, stream=True).raw).convert("RGB")

claims = [
    "a photo of two cats",
    "a photo of a fire truck",
]

# padding="max_length" matches SigLIP's training setup (fixed-length 64-token captions)
inputs = processor(text=claims, images=image, padding="max_length", return_tensors="pt").to(device)

with torch.no_grad():
    outputs = model(**inputs)
    # SigLIP uses an independent sigmoid per caption, not a softmax across candidates —
    # which is actually closer to what OOC thresholding needs: each claim judged on its own.
    probs = torch.sigmoid(outputs.logits_per_image)

print(f"{'Claim':30s} Match score")
for claim, p in zip(claims, probs[0]):
    print(f"{claim:30s} {p.item():.4f}")

## What this proves — and what to watch for

If "two cats" scored much higher than "fire truck," the whole chain works: GPU → Drive → packages → model → inference.

Two things to keep an eye on once you move to real Indic-language claims (your next milestone):

- **This was English.** SigLIP/CLIP-family models are trained overwhelmingly on English web image–text pairs. Section 5.2 of your synopsis names this directly as the untested transfer assumption — Hindi/Bengali/Marathi captions, and especially Roman-script transliteration, may not score anywhere near this cleanly. That gap is worth quantifying, not assuming.
- **This was "supported vs. wrong object,"** the easiest possible case. Real OOC pairs are a genuine image with a *plausible but false* caption (same kind of scene, wrong event/place/time) — a much harder signal to catch than "cats vs. fire truck."

Both become concrete once you swap in a handful of real (image, claim, label) triples, which is the next notebook.

## Save your work

`File → Save a copy in Drive` — or move this file into `MyDrive/IndicOOC/notebooks/` so it lives with the rest of the project.

## Next up

Once every cell above runs clean:
1. **Pilot data (O1, tiny slice)** — hand-pick ~10–15 real (image, claim, label) triples, a few per language, to replace the toy cats/fire-truck example
2. **Zero-shot baseline (O2-i)** — wrap Steps 5–6 into a proper loop with a decision threshold, run it over the pilot set, and get your first real macro-F1 number
3. Compare that number in English vs. Hindi/Bengali claims — this is your first genuine empirical result, weeks before Review 1

Let me know once this runs (or send the error if something breaks) and we'll build the pilot-data + baseline notebook next.